# 01 Data Merging

Notebook นี้รวมข้อมูล raw หลายตารางให้กลายเป็นไฟล์เดียวต่อ 1 match สำหรับใช้ใน EDA และ modeling


## 1. Import Library

โหลด `pandas` สำหรับอ่าน CSV, merge ตาราง, pivot ข้อมูล และบันทึกผลลัพธ์


In [ ]:
import pandas as pd

## 2. Load Player-Level Tables

อ่านตารางสถิติผู้เล่นจาก `MatchStatsTbl.csv` และตาราง mapping ผู้เล่นกับ match/champion จาก `SummonerMatchTbl.csv`


In [ ]:
stats_df = pd.read_csv('data/t1_raw/match_details/MatchStatsTbl.csv')
summoner_match_df = pd.read_csv('data/t1_raw/match_details/SummonerMatchTbl.csv')

## 3. Attach Match And Champion IDs

merge สถิติผู้เล่นกับข้อมูล `MatchFk` และ `ChampionFk` โดยใช้ `SummonerMatchFk` เทียบกับ `SummonerMatchId`


In [ ]:
merged_df = pd.merge(
    stats_df, 
    summoner_match_df[['SummonerMatchId', 'MatchFk', 'ChampionFk']], 
    left_on='SummonerMatchFk', 
    right_on='SummonerMatchId', 
    how='left'
)

## 4. Convert Player Rows Into One Match Row

เรียงผู้เล่นในแต่ละ match, สร้างหมายเลข `P1` ถึง `P10`, แล้ว pivot ให้หนึ่ง match เป็นหนึ่งแถว โดย column รายผู้เล่นจะมี suffix เช่น `_P1`, `_P2`


In [ ]:
merged_df = merged_df.sort_values(by=['MatchFk', 'SummonerMatchId'])

merged_df['Player_No'] = merged_df.groupby('MatchFk').cumcount() + 1
merged_df['Player_No'] = 'P' + merged_df['Player_No'].astype(str)

cols_to_pivot = ['ChampionFk', 'Lane', 'TotalGold', 'MinionsKilled', 'kills', 'deaths', 'assists', 'DmgDealt', 'DmgTaken', 'TurretDmgDealt', 'PrimaryKeyStone', 'SummonerSpell1', 'SummonerSpell2', 'Win']
df_1_row = merged_df.pivot(index='MatchFk', columns='Player_No', values=cols_to_pivot)

df_1_row.columns = [f"{col[0]}_{col[1]}" for col in df_1_row.columns]
df_1_row = df_1_row.reset_index()

df_1_row['TeamBlue_Win'] = df_1_row['Win_P1']

## 5. Remove Player-Level Win Columns

ลบ `Win_P*` หลังจาก pivot เพราะผลชนะของทีมจะใช้จากตาราง team-level แทน


In [ ]:
drop_cols = [col for col in df_1_row.columns if col.startswith('Win_P')]
df_1_row = df_1_row.drop(columns=drop_cols)

## 6. Add Team-Level Objective And Result Data

อ่าน `TeamMatchTbl.csv` แล้ว merge เข้ากับข้อมูลราย match เพื่อเพิ่ม objective counts และ target `BlueWin` / `RedWin`


In [ ]:
team_df = pd.read_csv('data/t1_raw/match_details/TeamMatchTbl.csv')

final_df = pd.merge(
    df_1_row, 
    team_df, 
    on='MatchFk', 
    how='inner'   
)

if 'TeamBlue_Win' in final_df.columns:
    final_df = final_df.drop(columns=['TeamBlue_Win'])

## 7. Add Match Metadata

อ่าน `MatchTbl.csv` แล้ว merge metadata เช่น `Patch`, `QueueType`, `RankFk`, และ `GameDuration` เข้ากับ dataset


In [ ]:
mdf = pd.read_csv("data/t1_raw/match_details/MatchTbl.csv")

merged = final_df.merge(mdf, left_on='MatchFk', right_on='MatchId', how='left')

if 'MatchId' in merged.columns:
    merged = merged.drop(columns='MatchId')

merged.head()

## 8. Create Working Copy

คัดลอก merged dataframe เป็น `df` เพื่อใช้ตรวจและ clean ข้อมูลก่อนบันทึก


In [ ]:
df = merged.copy()

## 9. Check Invalid Target Rows

ตรวจแถวที่ `BlueWin` และ `RedWin` มีค่าเท่ากัน ซึ่งเป็นกรณี target ไม่สอดคล้องกันสำหรับ binary classification


In [ ]:
df[df["BlueWin"] == df["RedWin"]]

## 10. Check Missing Champion IDs

ตรวจ missing values ใน `ChampionFk_P1` ถึง `ChampionFk_P10` เพราะแต่ละ match ควรมี champion ครบ 10 คน


In [ ]:
champ_fk_cols = [f"ChampionFk_P{p}" for p in range(1, 11)]
null_check = df[champ_fk_cols].isnull().sum()
null_check

## 11. Drop Rows With Missing Champion IDs

ลบ match ที่ champion ID ไม่ครบ เพื่อให้ข้อมูลผู้เล่นทั้งสองทีมสมบูรณ์ก่อนนำไปใช้ต่อ


In [ ]:
df = df.dropna(subset=champ_fk_cols).reset_index(drop=True)

## 12. Drop Unused Columns

ลบ column ที่ซ้ำกับข้อมูลผู้เล่นที่ pivot แล้ว หรือไม่ได้ใช้ใน transformed dataset เช่น ban/pick duplicate columns, `GameDuration`, `RankFk`, และ `TeamID`


In [ ]:
drop_cols = [
    "B1Champ", "B2Champ", "B3Champ", "B4Champ", "B5Champ",
    "R1Champ", "R2Champ", "R3Champ", "R4Champ", "R5Champ",
    "GameDuration", "RankFk", "TeamID",
]

df = df.drop(columns=drop_cols, errors="ignore")

## 13. Preview Final Data

แสดง dataframe หลัง merge และ clean เพื่อตรวจ column และจำนวนข้อมูลก่อน save


In [ ]:
df

## 14. Save Transformed Dataset

บันทึกไฟล์ `data/t2_transformed/merged_v1.csv` สำหรับใช้ใน notebook EDA และ modeling


In [ ]:
df.to_csv('data/t2_transformed/merged_v1.csv', index=False)